## Sentiment Analysis Mini Project

In [16]:
import torch
from datasets import load_dataset


In [17]:
dataset = load_dataset("stanfordnlp/imdb")
print(dataset)


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [18]:
dataset["train"][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [19]:
import re
from collections import Counter

def tokenize(text):
    text = text.lower()
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()


# Build vocab from training set
counter = Counter()
for example in dataset['train']:
    counter.update(tokenize(example['text']))
vocab_size = 30000
most_common = counter.most_common(vocab_size - 2)  # reserve 0 for pad, 1 for unk
vocab = {word: idx + 2 for idx, (word, _) in enumerate(most_common)}
vocab['<pad>'] = 0
vocab['<unk>'] = 1

def encode(text, max_len=200):
    tokens = tokenize(text)
    ids = [vocab.get(t, vocab['<unk>']) for t in tokens[:max_len]]
    ids += [vocab['<pad>']] * (max_len - len(ids))
    return ids

In [20]:
import torch
from torch.utils.data import Dataset, DataLoader

class IMDBDataset(Dataset):
    def __init__(self, hf_dataset, vocab, max_len=200):
        self.texts = hf_dataset['text']
        self.labels = hf_dataset['label']
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = encode(self.texts[idx], self.max_len)
        label = self.labels[idx]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(label, dtype=torch.float32)

In [21]:
from torch.utils.data import random_split

full_train = IMDBDataset(dataset['train'], vocab)
test_set = IMDBDataset(dataset['test'], vocab)

train_size = int(0.9 * len(full_train))
val_size = len(full_train) - train_size
train_set, val_set = random_split(full_train, [train_size, val_size])


In [22]:
batch_size = 64
trainloader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valloader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
testloader = DataLoader(test_set, batch_size=batch_size, shuffle=False)


In [23]:
x_batch, y_batch = next(iter(trainloader))
print(x_batch.shape, y_batch.shape)  # should be [64, 200] and [64]


torch.Size([64, 200]) torch.Size([64])


In [24]:
import torch.nn as nn

In [25]:
class TextClassifier(nn.Module):
  def __init__(self):
    super(TextClassifier,self).__init__()
    self.layer_1 = nn.Embedding(num_embeddings=30000,
                                embedding_dim=512,
                                padding_idx=0)
    self.layer_2 = nn.LSTM(input_size = 512, hidden_size=512, batch_first=True)
    self.layer_3 = nn.Dropout(0.5)
    self.layer_4 = nn.Sequential(
        nn.Linear(512,1),
        )
  def forward(self,x):
    x = self.layer_1(x)
    _, (h_n, c_n) = self.layer_2(x)
    x = h_n[0]   # fix this line
    x = self.layer_3(x)
    x = self.layer_4(x)
    return x





In [26]:
import torch.optim as optim

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = TextClassifier().to(device)

cuda


In [28]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

2.11.0+cu128
True
1


In [29]:
N_EPOCHS = 20

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

model = TextClassifier().to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters())
print(next(model.parameters()).device)

for epoch in range(N_EPOCHS):
    model.train()
    running_loss = 0
    num_correct = 0

    print(f"\nStarting Epoch {epoch + 1}/{N_EPOCHS}")

    for batch_idx, (inputs, labels) in enumerate(trainloader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs).squeeze(-1)

        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = (torch.sigmoid(outputs) > 0.5).float()
        num_correct += (preds == labels).sum().item()

        # Print progress
        if batch_idx % 50 == 0:
            print(
                f"Epoch {epoch + 1} | "
                f"Batch {batch_idx}/{len(trainloader)} | "
                f"Loss: {loss.item():.4f}"
            )

    train_loss = running_loss / len(trainloader)
    train_acc = num_correct / len(train_set)

    print(
        f"Epoch {epoch + 1} Complete | "
        f"Loss: {train_loss:.4f} | "
        f"Accuracy: {train_acc:.4f}"
    )

Using: cuda
cuda:0

Starting Epoch 1/20
Epoch 1 | Batch 0/352 | Loss: 0.6910
Epoch 1 | Batch 50/352 | Loss: 0.6891
Epoch 1 | Batch 100/352 | Loss: 0.6971
Epoch 1 | Batch 150/352 | Loss: 0.6988
Epoch 1 | Batch 200/352 | Loss: 0.6986
Epoch 1 | Batch 250/352 | Loss: 0.6476
Epoch 1 | Batch 300/352 | Loss: 0.6867
Epoch 1 | Batch 350/352 | Loss: 0.6803
Epoch 1 Complete | Loss: 0.6959 | Accuracy: 0.5100

Starting Epoch 2/20
Epoch 2 | Batch 0/352 | Loss: 0.7058
Epoch 2 | Batch 50/352 | Loss: 0.6783
Epoch 2 | Batch 100/352 | Loss: 0.6748
Epoch 2 | Batch 150/352 | Loss: 0.6667
Epoch 2 | Batch 200/352 | Loss: 0.7290
Epoch 2 | Batch 250/352 | Loss: 0.7172
Epoch 2 | Batch 300/352 | Loss: 0.7054
Epoch 2 | Batch 350/352 | Loss: 0.6866
Epoch 2 Complete | Loss: 0.6928 | Accuracy: 0.5328

Starting Epoch 3/20
Epoch 3 | Batch 0/352 | Loss: 0.6834
Epoch 3 | Batch 50/352 | Loss: 0.6573
Epoch 3 | Batch 100/352 | Loss: 0.6749
Epoch 3 | Batch 150/352 | Loss: 0.6542
Epoch 3 | Batch 200/352 | Loss: 0.6864
Epoch 

In [31]:
model.eval()
test_running_loss = 0
test_num_correct = 0
with torch.no_grad():
  for inputs, labels in testloader:
    inputs = inputs.to(device)
    labels = labels.to(device)
    outputs = model(inputs).squeeze(-1)
    loss = loss_fn(outputs,labels)
    test_running_loss +=loss.item()
    preds = (torch.sigmoid(outputs)>0.5).float()
    test_num_correct += (preds == labels).sum().item()
test_loss = test_running_loss / len(testloader)
test_acc = test_num_correct / len(test_set)

print(f'Test Loss: {test_loss:.4f}  Test Accuracy: {test_acc:.4f}')



Test Loss: 0.9411  Test Accuracy: 0.8087
